This is what I told Lylla:
https://data.cms.gov/provider-summary-by-type-of-service/medicare-part-d-prescribers/medicare-part-d-prescribers-by-provider-and-drug
Interested in: Who prescribes opioid addiction treatment medication and how has that changed over time? 

Based on some research I've done outside of this, I believe policy changes have made it easier for nurse practitioners and PA's especially to prescribe. And depending on how my laptop/python handles the amount of rows, I could either concentrate on NY state, a region, or nationally.

Hypothesis: Federal policy changes (2016, 2021, 2023) made it easier for nurse practitioners (NPs) and physician assistants (PAs) to prescribe, so their share should have grown.
- 2016- The Comprehensive Addiction and Recovery Act (CARA) let NPs and PAs get the federal waiver needed to prescribe buprenorphine for opioid use disorder
- 2021: HHS issued practice guidelines that exempted some clinicians from training and certification requirements for the waiver (helped NPs and PAs).
- 2023: MAT Act eliminated waiver requirement, so now most clinicians (including NPs and PAs) can prescripe buprenophine.


In [3]:
import pandas as pd
import requests
import os

In [4]:
#testing to make sure the API works
test_url = "https://data.cms.gov/data-api/v1/dataset/9552739e-3d05-4c1b-8eff-ecabf391e2e5/data"

params = {
    "filter[bup][condition][path]": "Gnrc_Name",
    "filter[bup][condition][operator]": "CONTAINS",
    "filter[bup][condition][value]": "Buprenorphine",
    "size": 3
}

#I learned about params = params from the scraping class. It is also in the documentation.
response = requests.get(test_url, params=params)
print("Status code:", response.status_code)   

test_df = pd.DataFrame(response.json())
test_df

Status code: 200


,Prscrbr_NPI,Prscrbr_Last_Org_Name,Prscrbr_First_Name,Prscrbr_City,Prscrbr_State_Abrvtn,Prscrbr_State_FIPS,Prscrbr_Type,Prscrbr_Type_Src,Brnd_Name,Gnrc_Name,...,Tot_Day_Suply,Tot_Drug_Cst,Tot_Benes,GE65_Sprsn_Flag,GE65_Tot_Clms,GE65_Tot_30day_Fills,GE65_Tot_Drug_Cst,GE65_Tot_Day_Suply,GE65_Bene_Sprsn_Flag,GE65_Tot_Benes
0,1003000142,Khalil,Rashid,Toledo,OH,39,Anesthesiology,Claim-Specialty,Belbuca,Buprenorphine Hcl,...,321,4388.84,,*,,,,,*,
1,1003001363,Stevens,Charles,El Centro,CA,06,Anesthesiology,Claim-Specialty,Buprenorphine,Buprenorphine,...,840,8844.58,,,30,30,8844.58,840,*,
2,1003001363,Stevens,Charles,El Centro,CA,06,Anesthesiology,Claim-Specialty,Buprenorphine-Naloxone,Buprenorphine Hcl/Naloxone Hcl,...,330,654.74,,#,,,,,*,


In [5]:
#now, I'll download all of the years, using the dataset ID for CMS data catolog
dataset_ids = {
    2013: "c6905d43-45de-470d-897c-9ed8e75e256d",
    2014: "0779bc8d-18dd-40b8-9d61-7addc8b0daf1",
    2015: "1d650894-8afe-4056-ba31-a85cb0e3cee6",
    2016: "25106f9d-0eb8-4ba7-b237-486ee87d910a",
    2017: "05f108dd-76c4-49f4-9fdc-788d8f4251ec",
    2018: "802fe556-311f-4962-8d75-d5f4ff405884",
    2019: "2a6705e6-7a1e-460c-ba22-35249a531918",
    2020: "7795fe20-e80e-435a-a9ed-d2d65e05feeb",
    2021: "f68114ed-f854-4ffc-9c6e-ed78b5e2f8d0",
    2022: "b101b457-ffa4-49bb-8fd9-27c1266086e2",
    2023: "e54db557-cd82-4e91-a0fe-61aad5865d69",
    2024: "9552739e-3d05-4c1b-8eff-ecabf391e2e5",
}

In [6]:
#I'll download one year. Note that the logic of this is in the API documentation.
import time

def get_year(year):
    url = f"https://data.cms.gov/data-api/v1/dataset/{dataset_ids[year]}/data"

    #Filter where generic name contains "Buprenorphine", because that is the medication I'm interested in
    filters = {
        "filter[bup][condition][path]": "Gnrc_Name",
        "filter[bup][condition][operator]": "CONTAINS",
        "filter[bup][condition][value]": "Buprenorphine",
    }

    #See how many rows match that
    stats = requests.get(url + "/stats", params=filters).json()
    total_rows = stats["found_rows"]
    print(f"{year}: {total_rows} rows to download")

    #Download 5,000 rows at a time
    all_rows = []
    for offset in range(0, total_rows, 5000):
        page_params = filters.copy()
        page_params["size"] = 5000
        page_params["offset"] = offset

        response = requests.get(url, params=page_params)
        all_rows = all_rows + response.json()
        time.sleep(1)   # short pause so CMS doesn't block us

    df = pd.DataFrame(all_rows)
    df["year"] = year
    return df

#test 2013 to make sure it worked
df_2013 = get_year(2013)
print(len(df_2013))
df_2013.head()

2013: 12556 rows to download
12556


,Prscrbr_NPI,Prscrbr_Last_Org_Name,Prscrbr_First_Name,Prscrbr_City,Prscrbr_State_Abrvtn,Prscrbr_State_FIPS,Prscrbr_Type,Prscrbr_Type_Src,Brnd_Name,Gnrc_Name,...,Tot_Drug_Cst,Tot_Benes,GE65_Sprsn_Flag,GE65_Tot_Clms,GE65_Tot_30day_Fills,GE65_Tot_Drug_Cst,GE65_Tot_Day_Suply,GE65_Bene_Sprsn_Flag,GE65_Tot_Benes,year
0,1003001363,Stevens,Charles,El Centro,CA,06,Anesthesiology,Claim-Specialty,Butrans,Buprenorphine,...,14974.74,16,,37,37,9706.65,1046,#,,2013
1,1003007469,Ozuna,Eduardo,Marble Falls,TX,48,Anesthesiology,Claim-Specialty,Suboxone,Buprenorphine Hcl/Naloxone Hcl,...,3950.85,,*,,,,,*,,2013
2,1003007741,Deppen,Krisanna,Grove City,OH,39,Family Practice,Claim-Specialty,Butrans,Buprenorphine,...,5161.69,,,0,0,0,0,#,,2013
3,1003007741,Deppen,Krisanna,Grove City,OH,39,Family Practice,Claim-Specialty,Suboxone,Buprenorphine Hcl/Naloxone Hcl,...,2422.62,,*,,,,,*,,2013
4,1003010133,Lepley,Jon,Lancaster,PA,42,Family Practice,NPPES-Specialty,Suboxone,Buprenorphine Hcl/Naloxone Hcl,...,6416.09,,,0,0,0,0,#,,2013


In [7]:
#making a new directory
os.makedirs("data", exist_ok=True)

#save every year of this data.
for year in dataset_ids:
    filename = f"data/buprenorphine_{year}.csv"

    #skip years I've already downloaded when re-running this
    if os.path.exists(filename):
        print(f"{year}: already saved, skipping")
        continue

    df_year = get_year(year)
    df_year.to_csv(filename, index=False)
    print(f"{year}: saved {len(df_year)} rows")

2013: already saved, skipping
2014: already saved, skipping
2015: already saved, skipping
2016: already saved, skipping
2017: already saved, skipping
2018: already saved, skipping
2019: already saved, skipping
2020: already saved, skipping
2021: already saved, skipping
2022: already saved, skipping
2023: already saved, skipping
2024: already saved, skipping


Per Lylla: Before I merge, make sure to run data.columns to ensure same columns or len(data.columns)
Make sure I have a sense of how big each data frame is
Check if merge worked by checking the length of the new data frame

In [20]:
#Load each year and check its size and columns
all_years = []

for year in range(2013, 2025):
    df_year = pd.read_csv(f"data/buprenorphine_{year}.csv",
                          #here, I fixed the dtype issue when I previously got an Dtype warning that the below columns had mixed types
                          dtype={"Prscrbr_State_FIPS": str, "Prscrbr_NPI": str})
    print(year, "rows:", len(df_year), "columns:", len(df_year.columns))
    all_years.append(df_year)

#all good on the number of columns

2013 rows: 12556 columns: 23
2014 rows: 16598 columns: 23
2015 rows: 18885 columns: 23
2016 rows: 21460 columns: 23
2017 rows: 24120 columns: 23
2018 rows: 30903 columns: 23
2019 rows: 38947 columns: 23
2020 rows: 42032 columns: 23
2021 rows: 43604 columns: 23
2022 rows: 45679 columns: 23
2023 rows: 48073 columns: 23
2024 rows: 51412 columns: 23


Now I have a lot of data saved in different files, but i need to combine and clean.

I am going to keep only the right buprenorphine I need (dropping other pain meds if there are any). 

And then I want to label the prescriber: there are NPs, PAs, and other. Should rerun this later too.

In [ ]:
#Note: This for loop that I used pefore was messing things up-- it added another full copy of my loop
#combining all the years
#for year in dataset_ids:
 #   df_year = pd.read_csv(f"data/buprenorphine_{year}.csv")
  #  all_years.append(df_year)

#Then concat
df = pd.concat(all_years)

total_of_years = sum(len(d) for d in all_years)
print("Rows if we add every year together:", total_of_years)
print("Rows in the combined table:", len(df))
df.head()
#so everything is good now, the duplicates are gone.

Rows if we add every year together: 394269
Rows in the combined table: 394269
Match: True
394269


,Prscrbr_NPI,Prscrbr_Last_Org_Name,Prscrbr_First_Name,Prscrbr_City,Prscrbr_State_Abrvtn,Prscrbr_State_FIPS,Prscrbr_Type,Prscrbr_Type_Src,Brnd_Name,Gnrc_Name,...,Tot_Drug_Cst,Tot_Benes,GE65_Sprsn_Flag,GE65_Tot_Clms,GE65_Tot_30day_Fills,GE65_Tot_Drug_Cst,GE65_Tot_Day_Suply,GE65_Bene_Sprsn_Flag,GE65_Tot_Benes,year
0,1003001363,Stevens,Charles,El Centro,CA,06,Anesthesiology,Claim-Specialty,Butrans,Buprenorphine,...,14974.74,16.0,NaN,37.0,37.0,9706.65,1046.0,#,NaN,2013
1,1003007469,Ozuna,Eduardo,Marble Falls,TX,48,Anesthesiology,Claim-Specialty,Suboxone,Buprenorphine Hcl/Naloxone Hcl,...,3950.85,NaN,*,NaN,NaN,NaN,NaN,*,NaN,2013
2,1003007741,Deppen,Krisanna,Grove City,OH,39,Family Practice,Claim-Specialty,Butrans,Buprenorphine,...,5161.69,NaN,NaN,0.0,0.0,0.00,0.0,#,NaN,2013
3,1003007741,Deppen,Krisanna,Grove City,OH,39,Family Practice,Claim-Specialty,Suboxone,Buprenorphine Hcl/Naloxone Hcl,...,2422.62,NaN,*,NaN,NaN,NaN,NaN,*,NaN,2013
4,1003010133,Lepley,Jon,Lancaster,PA,42,Family Practice,NPPES-Specialty,Suboxone,Buprenorphine Hcl/Naloxone Hcl,...,6416.09,NaN,NaN,0.0,0.0,0.00,0.0,#,NaN,2013


In [ ]:
#This should've been 12, but I got 36 since I ran it 3x on accident. So added this later to be extra sure. 
print("Number of years loaded:", len(all_years))

Number of years loaded: 12


In [24]:
#Looks good. See what that looks like.
df["Brnd_Name"].value_counts()

#Now I only want to keep addiction-related brands. I researched to find which brands are related for pain, not necessarily addiction. So I filtered those out.
addiction_brands = [
    "Buprenorphine-Naloxone",
    "Suboxone",
    "Buprenorphine Hcl",
    "Zubsolv",
    "Sublocade",
    "Bunavail",
    "Brixadi"]

#I'm finding- within brand column, checking if the brand is in my list. And outer df hecks the rows where that is true.
df_addiction = df[df["Brnd_Name"].isin(addiction_brands)]
print("Before:", len(df))
print("After:", len(df_addiction))


Before: 394269
After: 289146


In [ ]:
#double checking that total claims (what I want) is a number 
df_addiction["Tot_Clms"].dtype

dtype('int64')

In [40]:
df_addiction["Prscrbr_Type"].value_counts().head(50)

Prscrbr_Type
Family Practice                                                   65886
Nurse Practitioner                                                55747
Internal Medicine                                                 39385
Psychiatry                                                        38251
Physician Assistant                                               17615
Psychiatry & Neurology                                             7952
Anesthesiology                                                     7465
Emergency Medicine                                                 6784
Physical Medicine and Rehabilitation                               6367
General Practice                                                   6283
Addiction Medicine                                                 5651
Pain Management                                                    5511
Interventional Pain Management                                     4913
Obstetrics & Gynecology                            

From the above, I'm realizing that I can group by NPs, PAs, but then need an "other" category. Because there isn't a physician only category (ex. I see "dentist only" in run of the above that didn't include head. I also see student).

So for now, I will compare PAs and NPs to whole, as the non-physicians make up a very portion of the population. Assignment 2 can consider the breakdown more. And maybe a state by state analysis

In [ ]:
#see: theres a tiny amount of not clearly physicians, but I don't know if I should include them or not. Leaving it for now.
not_clearly_physicians = [
    "Specialist",
    "Student in an Organized Health Care Education/Training Program",
    "Certified Clinical Nurse Specialist",
    "Registered Nurse",
    "Certified Nurse Midwife",
    "Midwife",
    "Pharmacist",
    "Certified Registered Nurse Anesthetist (CRNA)",
]

not_clearly_physicians = df_addiction[df_addiction["Prscrbr_Type"].isin(not_clearly_physicians)]
not_clearly_physicians.groupby("year")["Prscrbr_NPI"].nunique()

year
2013     47
2014     54
2015     72
2016     91
2017     93
2018    125
2019    150
2020    169
2021    171
2022    188
2023    235
2024    225
Name: Prscrbr_NPI, dtype: int64

In [ ]:
# I'll label each prescriber into different groups
def prescriber_group(specialty):
    if specialty == "Nurse Practitioner":
        return "Nurse Practitioner"
    elif specialty == "Physician Assistant":
        return "Physician Assistant"
    else:
        return "Other"

#see the update
df_addiction["prescriber_group"].value_counts()

prescriber_group
Other    215784
NP        55747
PA        17615
Name: count, dtype: int64

Now, I'm ready for some analysis by years and my groups!

In [ ]:
#%otal prescriptions each year, from all prescribers
df_addiction.groupby("year")["Tot_Clms"].sum()

year
2013     532760
2014     626604
2015     708394
2016     833063
2017    1023932
2018    1262670
2019    1525442
2020    1716649
2021    1792349
2022    1872510
2023    1915244
2024    2001121
Name: Tot_Clms, dtype: int64

In [92]:
#Split the table I have into the three groups (according to prescriber types)
np_rows = df_addiction[df_addiction["prescriber_group"] == "NP"]
pa_rows = df_addiction[df_addiction["prescriber_group"] == "PA"]
other_rows = df_addiction[df_addiction["prescriber_group"] == "Other"]

#Now, add up prescriptions per year for each group. I'm looking at total claims, not just number of prescribers, because some people have multiple claims
#I think it means more if x group is writing more prescriptions, instead of just the total number of x group
np_by_year = np_rows.groupby("year")["Tot_Clms"].sum()
pa_by_year = pa_rows.groupby("year")["Tot_Clms"].sum()
other_by_year = other_rows.groupby("year")["Tot_Clms"].sum()

#Now, I put the results in one table
prescription_totals = pd.DataFrame({
    "NP": np_by_year,
    "PA": pa_by_year,
    "Other": other_by_year
})

prescription_totals

,NP,PA,Other
year,,,
2013,2885,966,528909
2014,3341,1610,621653
2015,3981,2697,701716
2016,4175,4033,824855
2017,17562,10158,996212
2018,90679,35654,1136337
2019,212322,72998,1240122
2020,360630,107984,1248035
2021,471573,141216,1179560


In [97]:
#But I want to see the shares too: lets try that

total = prescription_totals["NP"] + prescription_totals["PA"] + prescription_totals["Other"]

prescription_totals["NP_share"] = prescription_totals["NP"] / total * 100
prescription_totals["PA_share"] = prescription_totals["PA"] / total * 100
prescription_totals["NP_PA_share"] = prescription_totals["NP_share"] + prescription_totals["PA_share"]


prescription_totals.round(2)

,NP,PA,Other,Total,NP_share,PA_share,NP_PA_share
year,,,,,,,
2013,2885,966,528909,532760,0.54,0.18,0.72
2014,3341,1610,621653,626604,0.53,0.26,0.79
2015,3981,2697,701716,708394,0.56,0.38,0.94
2016,4175,4033,824855,833063,0.50,0.48,0.99
2017,17562,10158,996212,1023932,1.72,0.99,2.71
2018,90679,35654,1136337,1262670,7.18,2.82,10.01
2019,212322,72998,1240122,1525442,13.92,4.79,18.70
2020,360630,107984,1248035,1716649,21.01,6.29,27.30
2021,471573,141216,1179560,1792349,26.31,7.88,34.19


So my hypothesis seems correct for 2017: The share was flat through 2016, then jumped 3x for NPs and 2x for PAs in 2017. Since law was inacted in 2017.

And then, 2021 and 2023 doesn't show anything crazy. Thinking that may be because 2016 law applied specifically to NPs and PAs, but 2021 and 2023 also applied to physicians. So maybe nothing would impact the NP/PA share here. 

Another thing to try for V2: Does Buprenorphine for pain increase at the same rate? If not, that further validates my thesis, since the 2106 legislation was only for this addiction buprenorphine medication.

A finding from the above: 

NPs and PAs went from almost none of these opiod addiction prescriptions to nearly half. Together, they wrote under 1% of Medicare’s buprenorphine prescriptions for addiction from 2013 to 2016, and 46% in 2024 (NPs 36.3%, PAs 9.7%).

This shift also happened after the 2016 law. I can assume that since the law started being in effect in 2017, people just started prescribing in 2017 after meeting training requirements. And the ability to keep prescribing grew from there. 

In [90]:
# total prescriptions per year
total_by_year = df_addiction.groupby("year")["Tot_Clms"].sum()

# total prescriptions per year for NPs and PAs 
np_pa_rows = df_addiction[df_addiction["prescriber_group"].isin(["NP", "PA"])]
np_pa_by_year = np_pa_rows.groupby("year")["Tot_Clms"].sum()

total_growth = total_by_year[2024] - total_by_year[2016]
np_pa_growth = np_pa_by_year[2024] - np_pa_by_year[2016]

print("NP and PA share of prescription growth:", round(np_pa_growth / total_growth * 100, 1))

NP and PA share of prescription growth: 78.1


In [ ]:
#testing things out
df_addiction.groupby("year")["Prscrbr_NPI"].nunique().diff()

year
2013       NaN
2014     921.0
2015     679.0
2016     970.0
2017    1404.0
2018    2225.0
2019    2267.0
2020    2689.0
2021    1515.0
2022    1751.0
2023    1579.0
2024    2270.0
Name: Prscrbr_NPI, dtype: float64

In [ ]:
#see share of NPs/PAs in total prescribers
prescriber_counts = pd.DataFrame({
    "NP": np_counts,
    "PA": pa_counts,
    "Other": other_counts,
    "NP_PA": (np_counts+pa_counts)/(np_counts+pa_counts+other_counts) * 100
})

prescriber_counts

,NP,PA,Other,NP_PA
year,,,,
2013,56,40,6989,1.354975
2014,64,61,7881,1.561329
2015,75,78,8532,1.761658
2016,89,103,9463,1.988607
2017,480,246,10333,6.564789
2018,1586,591,11107,16.388136
2019,2883,929,11739,24.512893
2020,4137,1258,12845,29.577851
2021,5117,1535,13103,33.672488


One thing I noticed from above is that prior to 2017, NP counts is almost negligible. Then skyrocket in the years after that. That could be related to this data.